In [2]:
# -*- coding: utf-8 -*-
"""
Optimized Standard 2D Adapted SA-PINN (No RAR) - Fair Comparison Version
- FIXED: Periodic boundary loss properly evaluated explicitly at Y_MIN and Y_MAX (aligned with PINN).
- Exact X-boundaries are intrinsically satisfied via SA-PINN Boundary Lifting.
- Aligned with the strict LHS-based testing and timing protocol of 2D PINN.
"""

import os
import time
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import grad
from scipy.stats import qmc
from scipy.spatial import cKDTree

# =============================================================================
# Basic settings
# =============================================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01, 0.001, 0.0001]

X_MIN, X_MAX = -2.0, 2.0
Y_MIN, Y_MAX = -4.0, 4.0
T_FINAL = 1.0
L_BC, R_BC = -4.0, 2.0
H0_VALUE = 0.0

DEPTH, WIDTH = 5, 10
EPOCHS = 30000

# 采样点数
N_F = 4000
N_PER = 2000  # 用于周期边界的点对数量
N_I = 4000
# 总点数 = PDE内部点 + 周期下边界 + 周期上边界 + 初始点
TOTAL_POINTS = N_F + (2 * N_PER) + N_I

METHOD_NAME = "SAPINN2D"

# LHS testing settings
NUM_SAMPLES = 10000
LHS_SEED = 1234

# Repeated timing settings for T_eval
EVAL_WARMUP = 20
EVAL_REPEAT = 200

BASE_PATH = "."
SAVE_LHS_PREDICTION = True

# =============================================================================
# Utilities
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def source_f(x, y):
    return torch.cos(np.pi * x / 4.0) * torch.cos(np.pi * y / 4.0)

def u_init(x, y, mu):
    return 3.0 * torch.tanh(x / mu + y) - 1.0

def mean_std(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) <= 1:
        return np.nanmean(arr), 0.0
    return np.nanmean(arr), np.nanstd(arr, ddof=1)

# =============================================================================
# Network and 2D SA-PINN model
# =============================================================================
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width, depth):
        super().__init__()
        layers = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 2):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)
        
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

class AdaptedSAPINN2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.N_phi = MLP(2, 2, WIDTH, DEPTH)
        self.N_h = MLP(2, 1, WIDTH, DEPTH)

    def h(self, y, t):
        inp = torch.cat([y, t], dim=1)
        return H0_VALUE + t * self.N_h(inp)

    def outer(self, x, y):
        inp = torch.cat([x, y], dim=1)
        raw = self.N_phi(inp)
        phi_m = L_BC + (x - X_MIN) * raw[:, 0:1]
        phi_p = R_BC + (x - X_MAX) * raw[:, 1:2]
        return phi_m, phi_p

    def V_composite(self, x, y, t, mu):
        h_val = self.h(y, t)
        h_y = grad(h_val.sum(), y, create_graph=True)[0]
        S = 1.0 - h_y

        phi_m_xy, phi_p_xy = self.outer(x, y)
        phi_m_h, phi_p_h = self.outer(h_val, y)
        delta = phi_p_h - phi_m_h

        arg_l = (h_val - x) * delta * S / (2.0 * mu)
        arg_r = (x - h_val) * delta * S / (2.0 * mu)
        
        Q_l = delta * torch.sigmoid(-arg_l)
        Q_r = -delta * torch.sigmoid(-arg_r)

        left_branch = phi_m_xy + Q_l
        right_branch = phi_p_xy + Q_r
        
        return torch.where(x <= h_val, left_branch, right_branch)

    def forward(self, x, y, t, mu):
        base = self.V_composite(x, y, t, mu)
        x_l = torch.full_like(x, X_MIN)
        x_r = torch.full_like(x, X_MAX)
        
        u_l = self.V_composite(x_l, y, t, mu)
        u_r = self.V_composite(x_r, y, t, mu)
        
        ell_l = (X_MAX - x) / (X_MAX - X_MIN)
        ell_r = (x - X_MIN) / (X_MAX - X_MIN)
        
        # X-boundaries strictly enforced here
        return base + ell_l * (L_BC - u_l) + ell_r * (R_BC - u_r)

# =============================================================================
# Loss and Inference Evaluation
# =============================================================================
def compute_loss(model, X_f, Y_f, T_f, X_b, T_b, X_i, Y_i, U_i, mu):
    # 1. PDE Residual Loss
    u = model(X_f, Y_f, T_f, mu)
    
    grads_x = grad(u.sum(), X_f, create_graph=True)[0]
    grads_y = grad(u.sum(), Y_f, create_graph=True)[0]
    u_t = grad(u.sum(), T_f, create_graph=True)[0]
    
    u_xx = grad(grads_x.sum(), X_f, create_graph=True)[0]
    u_yy = grad(grads_y.sum(), Y_f, create_graph=True)[0]
    
    f = source_f(X_f, Y_f)
    pde_res = mu * (u_xx + u_yy) - u_t + u * (grads_x + grads_y) - f
    loss_pde = torch.mean(pde_res**2)

    # 2. Soft Periodic Boundary Loss (Aligned strictly with PINN boundaries)
    # y 必须带有 requires_grad=True，因为模型前向传播计算 h_y 时依赖它
    Y_bottom = torch.full_like(X_b, Y_MIN).requires_grad_(True)
    Y_top = torch.full_like(X_b, Y_MAX).requires_grad_(True)
    u_bottom = model(X_b, Y_bottom, T_b, mu)
    u_top = model(X_b, Y_top, T_b, mu)
    loss_per = torch.mean((u_bottom - u_top)**2)

    # 3. Initial Condition Loss
    T_0 = torch.zeros_like(X_i)
    u_initial = model(X_i, Y_i, T_0, mu)
    loss_ic = torch.mean((u_initial - U_i)**2)

    return loss_pde + loss_per + loss_ic

def eval_u(model, x_eval, y_eval, t_eval, mu):
    """
    Online SA-PINN reconstruction.
    CRITICAL: SA-PINN requires taking gradients with respect to y (to find h_y) 
    even during inference. We must use `enable_grad()`.
    """
    with torch.enable_grad():
        y_in = y_eval.detach().clone().requires_grad_(True)
        u_pred = model(x_eval, y_in, t_eval, mu)
    return u_pred

# =============================================================================
# LHS Testing Set and Error Computation
# =============================================================================
def get_target_col(df: pd.DataFrame) -> str:
    if "u" in df.columns: return "u"
    if "u0" in df.columns: return "u0"
    return df.columns[-1]

def load_true_solution(mu):
    mu_id = round(-math.log10(mu))
    filename = f"2d_U0_all_t_u_x_y_t_mu{mu_id}_101_101_101_Mathematica_620.csv"
    path = os.path.join(BASE_PATH, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find true solution file: {filename}")

    df = pd.read_csv(path)
    df.columns = [str(col).lower().strip() for col in df.columns]
    df = df.sort_values(by=["t", "x", "y"]).reset_index(drop=True)
    return df, filename

def generate_lhs_indices(df_true, mu):
    total_points = len(df_true)
    if total_points < NUM_SAMPLES:
        raise ValueError(f"Reference grid only has {total_points} points.")

    t_min, t_max = df_true["t"].min(), df_true["t"].max()
    x_min, x_max = df_true["x"].min(), df_true["x"].max()
    y_min, y_max = df_true["y"].min(), df_true["y"].max()
    all_points = df_true[["t", "x", "y"]].values
    kdtree = cKDTree(all_points)

    selected = []
    used = set()
    batch_id = 0

    while len(selected) < NUM_SAMPLES and batch_id < 100:
        sampler = qmc.LatinHypercube(d=3, seed=LHS_SEED + batch_id)
        lhs_sample = sampler.random(n=NUM_SAMPLES)
        lhs_scaled = qmc.scale(lhs_sample, [t_min, x_min, y_min], [t_max, x_max, y_max])
        _, candidate_indices = kdtree.query(lhs_scaled)

        for idx in candidate_indices:
            idx = int(idx)
            if idx not in used:
                used.add(idx)
                selected.append(idx)
                if len(selected) == NUM_SAMPLES:
                    break
        batch_id += 1

    if len(selected) < NUM_SAMPLES:
        remaining = np.setdiff1d(np.arange(total_points), np.asarray(selected, dtype=int), assume_unique=False)
        fill = np.random.default_rng(LHS_SEED).choice(remaining, size=NUM_SAMPLES - len(selected), replace=False)
        selected.extend([int(i) for i in fill])

    sample_indices = np.asarray(selected, dtype=int)
    np.save(f"2d_LHS_sample_indices_mu{mu:.0e}.npy", sample_indices)
    return sample_indices

def build_or_load_lhs_test_set_from_true(mu):
    df_true, filename = load_true_solution(mu)
    index_file = f"2d_LHS_sample_indices_mu{mu:.0e}.npy"

    if os.path.exists(index_file):
        sample_indices = np.load(index_file)
        valid = (len(sample_indices) == NUM_SAMPLES and len(np.unique(sample_indices)) == NUM_SAMPLES and np.max(sample_indices) < len(df_true))
        if valid:
            print(f"[mu={mu}] Loaded valid LHS indices from {index_file}.")
        else:
            sample_indices = generate_lhs_indices(df_true, mu)
    else:
        sample_indices = generate_lhs_indices(df_true, mu)
        print(f"[mu={mu}] Generated and saved {len(sample_indices)} LHS indices.")

    t_lhs_np = df_true.iloc[sample_indices]["t"].values.reshape(-1, 1)
    x_lhs_np = df_true.iloc[sample_indices]["x"].values.reshape(-1, 1)
    y_lhs_np = df_true.iloc[sample_indices]["y"].values.reshape(-1, 1)
    true_col = get_target_col(df_true)
    true_lhs_np = df_true.iloc[sample_indices][true_col].values.reshape(-1)

    return {
        "t_lhs": torch.tensor(t_lhs_np, dtype=torch.float32, device=DEVICE),
        "x_lhs": torch.tensor(x_lhs_np, dtype=torch.float32, device=DEVICE),
        "y_lhs": torch.tensor(y_lhs_np, dtype=torch.float32, device=DEVICE),
        "t_lhs_np": t_lhs_np,
        "x_lhs_np": x_lhs_np,
        "y_lhs_np": y_lhs_np,
        "true_lhs_np": true_lhs_np,
        "n_test": len(sample_indices)
    }

def compute_error(true_u, pred_u):
    diff = pred_u - true_u
    e2 = np.linalg.norm(diff) / np.linalg.norm(true_u)
    einf = np.max(np.abs(diff))
    return e2, einf

# =============================================================================
# Main Program
# =============================================================================
if __name__ == "__main__":
    print("\n" + "=" * 80)
    print(f"Starting 2D {METHOD_NAME} benchmark with strict LHS-based T_eval and error")
    print("=" * 80 + "\n")

    lhs_data = {mu: build_or_load_lhs_test_set_from_true(mu) for mu in MU_LIST}
    metrics = {mu: [] for mu in MU_LIST}

    for mu in MU_LIST:
        print("\n" + "=" * 80)
        print(f"Starting 2D {METHOD_NAME} for mu={mu}")
        print("=" * 80)

        data_mu = lhs_data[mu]
        x_eval_lhs = data_mu["x_lhs"]
        y_eval_lhs = data_mu["y_lhs"]
        t_eval_lhs = data_mu["t_lhs"]
        true_lhs_np = data_mu["true_lhs_np"]
        n_test = data_mu["n_test"]

        for seed in SEEDS:
            print("\n" + "-" * 80)
            print(f"Running mu={mu}, seed={seed}")
            print("-" * 80)

            set_seed(seed)

            # 1. Collocation Points
            X_f = torch.tensor(np.random.uniform(X_MIN, X_MAX, (N_F, 1)), dtype=torch.float32, device=DEVICE).requires_grad_(True)
            Y_f = torch.tensor(np.random.uniform(Y_MIN, Y_MAX, (N_F, 1)), dtype=torch.float32, device=DEVICE).requires_grad_(True)
            T_f = torch.tensor(np.random.uniform(0, T_FINAL, (N_F, 1)), dtype=torch.float32, device=DEVICE).requires_grad_(True)

            # 周期边界点 (只在 X 和 T 上随机采样，y 将在 compute_loss 中硬赋值为 -4 和 4)
            X_b = torch.tensor(np.random.uniform(X_MIN, X_MAX, (N_PER, 1)), dtype=torch.float32, device=DEVICE)
            T_b = torch.tensor(np.random.uniform(0, T_FINAL, (N_PER, 1)), dtype=torch.float32, device=DEVICE)

            # 初始点
            X_i = torch.tensor(np.random.uniform(X_MIN, X_MAX, (N_I, 1)), dtype=torch.float32, device=DEVICE)
            Y_i = torch.tensor(np.random.uniform(Y_MIN, Y_MAX, (N_I, 1)), dtype=torch.float32, device=DEVICE).requires_grad_(True)
            U_i = u_init(X_i, Y_i, mu).detach()

            model = AdaptedSAPINN2D().to(DEVICE)
            optimizer = optim.Adam(model.parameters(), lr=1e-3)
            
            # Loss list avoids CPU syncing during training
            loss_list = []

            model.train()
            if torch.cuda.is_available(): torch.cuda.synchronize()
            train_start = time.perf_counter()

            for epoch in range(EPOCHS):
                optimizer.zero_grad(set_to_none=True)
                loss = compute_loss(model, X_f, Y_f, T_f, X_b, T_b, X_i, Y_i, U_i, mu)
                loss.backward()
                optimizer.step()
                
                loss_list.append(loss.detach())

            if torch.cuda.is_available(): torch.cuda.synchronize()
            T_train = time.perf_counter() - train_start

            loss_history = torch.stack(loss_list).cpu().numpy().astype(np.float64)
            e_loss = float(loss_history[-1])

            T_train_per_iter_ms = T_train * 1e3 / EPOCHS
            T_train_per_iter_point_us = T_train * 1e6 / (EPOCHS * TOTAL_POINTS)

            print(f" > Trained: epochs={EPOCHS}, points={TOTAL_POINTS}, T_train={T_train:.2f}s, e_loss={e_loss:.3e}")
            np.save(f"2d_{METHOD_NAME}_loss_history_mu{mu:.0e}_seed{seed}.npy", loss_history)

            model.eval()

            # Warm-up outside of T_eval
            for _ in range(EVAL_WARMUP):
                _ = eval_u(model, x_eval_lhs, y_eval_lhs, t_eval_lhs, mu)
            if torch.cuda.is_available(): torch.cuda.synchronize()

            # Timed Evaluation Loop
            eval_start = time.perf_counter()
            for _ in range(EVAL_REPEAT):
                _ = eval_u(model, x_eval_lhs, y_eval_lhs, t_eval_lhs, mu)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            T_eval = (time.perf_counter() - eval_start) / EVAL_REPEAT

            # Error computation once, outside T_eval
            u_eval_tensor = eval_u(model, x_eval_lhs, y_eval_lhs, t_eval_lhs, mu)
            u_pred_lhs = u_eval_tensor.detach().cpu().numpy().reshape(-1)
            e2, einf = compute_error(true_lhs_np, u_pred_lhs)

            T_total = T_train + T_eval

            metrics[mu].append({
                "Seed": seed,
                "N_test": n_test,
                "e_loss": e_loss,
                "e2": e2,
                "einf": einf,
                "T_train": T_train,
                "T_eval": T_eval,
                "T_total": T_total,
                "T_train_per_iter_ms": T_train_per_iter_ms,
                "T_train_per_iter_point_us": T_train_per_iter_point_us,
                "total_trained_steps": EPOCHS,
                "total_point_steps": EPOCHS * TOTAL_POINTS,
                "final_residual_points": TOTAL_POINTS,
                "eval_warmup": EVAL_WARMUP,
                "eval_repeat": EVAL_REPEAT
            })

            print(f"    -> [mu={mu}] N_test={n_test}, T_eval={T_eval:.6e}s, e2={e2:.3e}, einf={einf:.3e}")

            if SAVE_LHS_PREDICTION:
                df_lhs_pred = pd.DataFrame({
                    "t": data_mu["t_lhs_np"].reshape(-1),
                    "x": data_mu["x_lhs_np"].reshape(-1),
                    "y": data_mu["y_lhs_np"].reshape(-1),
                    "u": u_pred_lhs
                })
                df_lhs_pred.to_csv(f"2d_{METHOD_NAME}_U0_predicted_LHS_mu{mu:.0e}_seed{seed}.csv", index=False)

# =============================================================================
# Summary tables
# =============================================================================
print("\n" + "=" * 80)
print("ALL SEEDS COMPLETED. GENERATING SUMMARY TABLES.")
print("=" * 80 + "\n")

for mu in MU_LIST:
    dfm = pd.DataFrame(metrics[mu])
    dfm.to_csv(f"2d_{METHOD_NAME}_mu{mu:.0e}_Metrics_Summary.csv", index=False)

    cols = ["e_loss", "e2", "einf", "T_train", "T_eval", "T_total", 
            "T_train_per_iter_ms", "T_train_per_iter_point_us", 
            "total_trained_steps", "total_point_steps", "final_residual_points", "N_test"]

    stats = {c: mean_std(dfm[c].values) for c in cols}

    print(f"### Results for 2D {METHOD_NAME}, mu={mu} [Mean +/- Sample Std] ###")
    print(f"N_test: {stats['N_test'][0]:.0f} +/- {stats['N_test'][1]:.0f}")
    print(f"e_loss: {stats['e_loss'][0]:.3e} +/- {stats['e_loss'][1]:.3e}")
    print(f"e_2: {stats['e2'][0]:.3e} +/- {stats['e2'][1]:.3e}")
    print(f"e_inf: {stats['einf'][0]:.3e} +/- {stats['einf'][1]:.3e}")
    print(f"T_train (s): {stats['T_train'][0]:.2f} +/- {stats['T_train'][1]:.2f}")
    print(f"T_eval (s): {stats['T_eval'][0]:.6e} +/- {stats['T_eval'][1]:.6e}")
    print(f"T_total (s): {stats['T_total'][0]:.2f} +/- {stats['T_total'][1]:.2f}")
    print(f"T_train/iter (ms): {stats['T_train_per_iter_ms'][0]:.4f} +/- {stats['T_train_per_iter_ms'][1]:.4f}")
    print(f"T_train/(iter*pt) (us): {stats['T_train_per_iter_point_us'][0]:.4f} +/- {stats['T_train_per_iter_point_us'][1]:.4f}")
    print(f"Optimization steps: {stats['total_trained_steps'][0]:.0f} +/- {stats['total_trained_steps'][1]:.0f}")
    print(f"Loss point-steps: {stats['total_point_steps'][0]:.0f} +/- {stats['total_point_steps'][1]:.0f}\n")


Starting 2D SAPINN2D benchmark with strict LHS-based T_eval and error

[mu=0.01] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-02.npy.
[mu=0.001] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-03.npy.
[mu=0.0001] Loaded valid LHS indices from 2d_LHS_sample_indices_mu1e-04.npy.

Starting 2D SAPINN2D for mu=0.01

--------------------------------------------------------------------------------
Running mu=0.01, seed=33
--------------------------------------------------------------------------------
 > Trained: epochs=30000, points=12000, T_train=5723.84s, e_loss=2.291e+00
    -> [mu=0.01] N_test=10000, T_eval=8.190377e-03s, e2=6.714e-01, einf=7.284e+00

--------------------------------------------------------------------------------
Running mu=0.01, seed=99
--------------------------------------------------------------------------------
 > Trained: epochs=30000, points=12000, T_train=5714.21s, e_loss=7.015e+00
    -> [mu=0.01] N_test=10000, T_eval=8.220998e-03s, e2=6.